<a href="https://colab.research.google.com/github/bsheese/387/blob/main/19_onramp/19_1_ML_Foundations/19_1_3_One_Neuron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The On-Ramp: Part 3
## One Neuron

---

## What This Notebook Is About

In 19_0 we said that the last layer of the Pets model turns each photo's 512 features into 4 scores, one per breed, and left it at that. This notebook shows exactly what that calculation is. It turns out to be simple: each score is a **weighted sum** of the features, plus one extra number. A weighted sum followed by a function that turns it into an output is what deep learning calls a **neuron**, and the last layer of our model is four neurons side by side.

We get there in steps:

1. What a weighted sum is, computed by hand and then with NumPy (the **dot product**)
2. A model with one neuron: **logistic regression**, trained on the 512 features to tell Birmans from Ragdolls
3. Reproducing one of its predictions by hand: a weighted sum, then the **sigmoid** function
4. The Pets model's own last layer: four weighted sums at once, which is a **matrix multiplication**, and the shape rule that goes with it

**Optional reference.** If you took CS 377, you have seen these ideas before in [18_2_1](https://github.com/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_1_LogReg_Intro_w_Titanic.ipynb), [18_2_3](https://github.com/bsheese/cs377/blob/main/18_classification/18_2_LogisticRegression/18_2_3_LogReg_to_NeuralNetworks.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

preds = pd.read_csv(fetch('pets4_predictions.csv'))
saved = np.load(fetch('pets4_features.npz'))
features = saved['features']
split = saved['split']
breed = saved['breed']
BREEDS = ['Birman', 'Ragdoll', 'basset_hound', 'beagle']
print('features:', features.shape)

---

## Section 1: A Weighted Sum

Here is a small made-up example. Suppose a model looks at only three features of an animal, and has learned one **weight** for each feature and one extra number called the **bias**. To get its score, multiply each feature by its weight, add the results, and add the bias:

In [ ]:
x = np.array([2.0, 0.5, 1.0])     # three features of one example
w = np.array([1.5, -2.0, 0.25])   # one weight per feature
b = -0.5                          # the bias

score = x[0] * w[0] + x[1] * w[1] + x[2] * w[2] + b
print('score:', score)

That is 2.0 × 1.5 + 0.5 × (−2.0) + 1.0 × 0.25 − 0.5 = 3.0 − 1.0 + 0.25 − 0.5 = 1.75.

- A **positive weight** means a larger value of that feature pushes the score up.
- A **negative weight** means a larger value pushes the score down.
- A weight near **zero** means that feature barely matters.
- The **bias** shifts the score up or down whatever the features are.

Writing out every term is fine for three features but not for 512. Multiplying two arrays of the same length element by element and adding up the results is called the **dot product**, and NumPy has an operator for it, `@`:

In [ ]:
print('element by element:', x * w)
print('added up:          ', (x * w).sum())
print('with @:            ', x @ w)
print('score = x @ w + b: ', x @ w + b)

The dot product only makes sense when the two arrays have the same length, because every feature needs exactly one weight. If the lengths don't match, NumPy raises an error rather than guessing:

In [ ]:
try:
    np.array([1.0, 2.0, 3.0]) @ np.array([1.0, 2.0])
except ValueError as e:
    print('ValueError:', e)

Errors like that one, about shapes that don't match, are among the most common errors in deep learning code. When you see one, check the shape of each array. That habit will save you a lot of time later in the course.

---

## Section 2: A Neuron That Tells Birmans from Ragdolls

Now we use real features. To keep things to one score, we take only the two cat breeds and ask a yes/no question: is this photo a Ragdoll? The model that answers a yes/no question with a weighted sum is called **logistic regression**. scikit-learn has it built in.

First, select the Birman and Ragdoll photos from the training and test sets. `|` combines two masks with "or", and `&` with "and".

In [ ]:
is_cat = (breed == 'Birman') | (breed == 'Ragdoll')
train = is_cat & (split == 'train')
test = is_cat & (split == 'test')

X_train = features[train]
y_train = breed[train] == 'Ragdoll'    # True for Ragdoll, False for Birman
X_test = features[test]
y_test = breed[test] == 'Ragdoll'

print('training:', X_train.shape, '  test:', X_test.shape)

`X_train` has 200 rows (100 Birmans and 100 Ragdolls) and 512 columns. `.fit` learns one weight per column plus a bias from the training photos. (How it finds good weights is the subject of unit 20.) `.score` gives the accuracy.

In [ ]:
neuron = LogisticRegression(max_iter=5000)
neuron.fit(X_train, y_train)

print(f'training accuracy: {neuron.score(X_train, y_train):.3f}')
print(f'test accuracy:     {neuron.score(X_test, y_test):.3f}')

On the test photos, this single neuron tells Birmans from Ragdolls about 9 times in 10. Telling this pair apart is the hard part of the four-breed problem. For comparison, the four-breed model from 19_0 got 84 of the 100 test Birmans and Ragdolls right. (The training accuracy of 1.0 against a lower test accuracy is the gap from 19_1_1.)

The learned weights are stored in `coef_` and the bias in `intercept_`. There is one weight for each of the 512 features.

In [ ]:
w = neuron.coef_[0]
b = neuron.intercept_[0]
print('weights:', w.shape)
print('first five weights:', w[:5].round(3))
print('bias:', round(b, 3))

---

## Section 3: One Prediction by Hand

`predict_proba` gives the model's probability for each answer. Here it is for the first test photo. The two columns are the probability of `False` (Birman) and of `True` (Ragdoll).

In [ ]:
x = X_test[0]
print('true breed:', 'Ragdoll' if y_test[0] else 'Birman')
print('predict_proba:', neuron.predict_proba(x.reshape(1, -1)).round(4))

(`predict_proba` expects a table of photos, so `x.reshape(1, -1)` turns the single photo's 512 features into a table with one row.)

Now let's get the same probability ourselves. Step one is the weighted sum of the photo's features, exactly as in Section 1:

In [ ]:
z = x @ w + b
print('weighted sum z:', round(z, 4))

The weighted sum can be any number, positive or negative. To turn it into a probability between 0 and 1, logistic regression passes it through the **sigmoid** function:

$$\text{sigmoid}(z) = \frac{1}{1 + e^{-z}}$$

Here $e$ is the number 2.718..., and $e^{-z}$ means $e$ raised to the power $-z$. `np.exp(a)` computes $e^{a}$. What matters about $e^{-z}$ is how it behaves:

- When $z$ is a large positive number, $e^{-z}$ is close to 0, so sigmoid is close to 1/1 = 1.
- When $z$ is 0, $e^{0} = 1$, so sigmoid is 1/2 = 0.5.
- When $z$ is a large negative number, $e^{-z}$ is huge, so sigmoid is close to 0.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for value in [-6, -2, 0, 2, 6]:
    print(f'sigmoid({value:2}) = {sigmoid(value):.4f}')

zs = np.linspace(-8, 8, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(zs, sigmoid(zs))
plt.axhline(0.5, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.xlabel('weighted sum z')
plt.ylabel('sigmoid(z)')
plt.title('The sigmoid function')
plt.show()

Sigmoid squeezes any number into the range from 0 to 1. A weighted sum of 0 becomes a probability of 0.5, which is exactly on the fence. Positive sums give probabilities above 0.5 (Ragdoll is more likely) and negative sums give probabilities below 0.5 (Birman is more likely).

Now apply it to the weighted sum for our photo:

In [ ]:
print('by hand:        ', round(sigmoid(z), 4))
print('predict_proba:  ', round(neuron.predict_proba(x.reshape(1, -1))[0, 1], 4))

They match. Everything logistic regression does when it makes a prediction is these two steps: a weighted sum, then sigmoid.

### This is a neuron

That two-step calculation is what deep learning calls an artificial **neuron**:

```
 features         weights
   x[0]  ──────── w[0] ──┐
   x[1]  ──────── w[1] ──┤
   x[2]  ──────── w[2] ──┼──►  z = weighted sum + b  ──►  activation(z)  ──►  output
   ...                   │
   x[511] ─────── w[511] ┘
```

The function applied at the end is called the **activation function**. Here it is sigmoid; later in the course you will meet others. The idea dates from Frank Rosenblatt's perceptron in 1958, and the name comes from a loose analogy with nerve cells, which combine many incoming signals and fire when the total is high enough. Every network in this course is built from units like this one.

---

## Section 4: The Pets Model's Last Layer Is Four Neurons

The last layer of the Pets model does the same thing, except that it gives four scores instead of one. It has one set of 512 weights and one bias for **each** breed. The next cell loads the trained last layer that 19_0 saved. It holds a weight array of shape (4, 512), with one row of weights per breed, and a bias array with 4 numbers.

In [ ]:
head = torch.load(fetch('pets4_head.pt'))
W = head['weight'].numpy()
bias = head['bias'].numpy()
print('W:', W.shape, '  bias:', bias.shape)

The Birman score of a photo is the dot product of its features with row 0 of `W`, plus `bias[0]`. The Ragdoll score uses row 1, and so on. Here are the four scores for the first photo in the file, computed one at a time:

In [ ]:
x = features[0]
print('photo:', saved['filename'][0], '  true breed:', breed[0])
for i, name in enumerate(BREEDS):
    print(f'{name:13} score: {x @ W[i] + bias[i]:7.3f}')

These should be the scores 19_0_2 saved for this photo:

In [ ]:
print(preds.loc[0, ['score_Birman', 'score_Ragdoll', 'score_basset_hound', 'score_beagle']].to_numpy().astype(float).round(3))

They match. The highest score is Birman's, so the model predicts Birman for this photo.

### Four weighted sums at once: matrix multiplication

Computing the four scores in a loop works, but `@` can do all four at once. When the left side is a two-dimensional array (a **matrix**), `W @ x` computes the dot product of **every row** of `W` with `x`:

In [ ]:
scores = W @ x + bias
print(scores.round(3))

This is **matrix multiplication**, and it's the main calculation inside every neural network. The rule about shapes is the one that matters most:

> `(4, 512) @ (512,)` gives `(4,)`. The **inner** sizes (512 and 512) must match, because each row of weights needs one feature for each weight. The **outer** size (4) is what's left: one result per row of weights.

The same works for many photos at once. `features` has shape (800, 512), with one photo per row. To get every photo's four scores, each photo's row has to be combined with each breed's row of weights. `W.T` is `W` **transposed**, with its rows and columns swapped, so its shape is (512, 4) and the inner sizes line up:

In [ ]:
print('features:', features.shape, '  W.T:', W.T.shape)

all_scores = features @ W.T + bias
print('all_scores:', all_scores.shape)

saved_scores = preds[['score_Birman', 'score_Ragdoll', 'score_basset_hound', 'score_beagle']].to_numpy()
print('largest difference from the saved scores:', np.abs(all_scores - saved_scores).max().round(6))

`(800, 512) @ (512, 4)` gives `(800, 4)`: the inner sizes match (512 and 512), and the result has one row per photo and one column per breed. All 3,200 scores agree with the saved ones, apart from rounding in the last decimal places. (Adding `bias`, which has shape (4,), to a result of shape (800, 4) adds the same four biases to every row. NumPy calls this **broadcasting**.)

If you try it the other way round, the inner sizes don't match and NumPy refuses:

In [ ]:
try:
    features @ W
except ValueError as e:
    print('ValueError:', e)

"Check the inner sizes" will be the first thing to do with almost every shape error in this course.

---

## Summary

- A **weighted sum** multiplies each feature by its **weight**, adds the results and adds a **bias**. Multiplying matching elements and adding them up is the **dot product**, written `x @ w` in NumPy. It needs the two arrays to have the same length.
- **Logistic regression** answers a yes/no question with one weighted sum followed by the **sigmoid** function, which squeezes any number into a probability between 0 and 1 (0 becomes 0.5). Trained on the 512 features, it tells Birmans from Ragdolls about 9 times in 10. Its `predict_proba` can be reproduced exactly by hand.
- A weighted sum followed by an **activation function** is an artificial **neuron**. Logistic regression is a single neuron with a sigmoid activation.
- The Pets model's last layer is four neurons, one per breed: a (4, 512) weight matrix and 4 biases. Computing all four scores at once is a **matrix multiplication**, `W @ x + bias`.
- **The shape rule**: in `A @ B` the inner sizes must match, and the outer sizes give the shape of the result. `(800, 512) @ (512, 4)` gives `(800, 4)`.

**Next:** 19_1_4 shows how the last layer's four scores become four probabilities (softmax), and then finds a simple problem that no single neuron can solve, which is why networks have more than one layer.